# Reference attention under free generation (twelve carriers), Kaggle T4

Same Tesla T4, same Qwen3-4B revision, FP16 weights, greedy decoding, 384 plan / 48 answer tokens. Same nine cases, frozen A artifacts, generic support, prompts and positions. Oracle diagnostic, not a method. No adapter.

Branches (all free generation, scored normally):
- **tk_tv**, **fk_tv**, **fk_fv**: ordinary generation (should reproduce the K/V swap run)
- **tk_tv_refattn**: two-row lockstep [fk_fv, tk_tv]. Reference attention weights on tk_tv's own values during B's prompt processing and every decoding step. The next token is the **intervention's** argmax and is fed to both rows; the reference only follows that history. Saved reference text is never fed.
- **tk_tv_selfattn**: lockstep [tk_tv, tk_tv] replaying its own attention (control)

The answer pass uses each branch's own plan with the same procedure.

Per-case checks on the incremental path: the self-replay rows and the reference row must match an ordinary forward on the same generated history (mean KL <= 0.01, argmax agreement >= 0.98, identical label decisions), replacement must hit every layer of every forward, and the saved caches must be unchanged. The run stops at the first failing case.

**Cell 4** runs the first case only; read its checks. **Cell 5** runs the remaining eight (resumes).

Upload `src/`, `configs/`, `scripts/`, `data/plan4_timing12.jsonl`, and the folder `results_stage_plan4_refattn12/` (it already holds `a_artifacts_12.jsonl`, `recon12_check_summary.json`, `kvmix12_evaluations.jsonl`, `summary.json`).

Download `/kaggle/working/results_stage_plan4_refgen12`.

In [ ]:
import os, json, shutil, sys
from pathlib import Path

os.environ["LATENT_RELAY_INFER_DTYPE"] = "fp16"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

ON_KAGGLE = Path("/kaggle/working").exists()
WORK = Path("/kaggle/working") if ON_KAGGLE else Path.cwd()

candidates = [Path.cwd()]
if Path("/kaggle/input").is_dir():
    for root, dirs, files in os.walk("/kaggle/input"):
        dirs[:] = [d for d in dirs if d not in {".git", ".pytest_cache"}]
        if "stage_plan4_refgen12.py" in files and Path(root).name == "src":
            candidates.append(Path(root).parent)

REPO = None
for cand in candidates:
    if (cand / "src" / "stage_plan4_refgen12.py").is_file() and (cand / "data" / "plan4_timing12.jsonl").is_file():
        REPO = cand
        break
if REPO is None:
    raise FileNotFoundError("Need src/stage_plan4_refgen12.py and data/plan4_timing12.jsonl.")

if ON_KAGGLE:
    dest = WORK / "latent-relay-audit"
    if str(REPO).startswith("/kaggle/input") or not (dest / "src" / "stage_plan4_refgen12.py").is_file():
        shutil.copytree(REPO, dest, dirs_exist_ok=True)
    REPO = dest
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print("REPO", REPO.resolve())
for name in ("a_artifacts_12.jsonl", "recon12_check_summary.json", "kvmix12_evaluations.jsonl", "summary.json"):
    print("results_stage_plan4_refattn12", name, (REPO / "results_stage_plan4_refattn12" / name).is_file())

In [ ]:
import subprocess, sys
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers==4.57.3", "tokenizers==0.22.2", "huggingface_hub==0.36.2",
    "safetensors==0.7.0", "accelerate==1.13.0", "datasets==4.4.1",
])
import torch, transformers
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
print("cuda", torch.cuda.is_available(), gpu)
print("transformers", transformers.__version__)
assert transformers.__version__.startswith("4.57")
if gpu is not None and "T4" not in gpu:
    raise RuntimeError(f"Stay on the locked Tesla T4. Got {gpu}.")

In [ ]:
subprocess.check_call([sys.executable, "scripts/fetch_upstream.py"])
print("Dest /kaggle/working/results_stage_plan4_refgen12. First case, then the rest.")

In [ ]:
rc = subprocess.call([sys.executable, "-m", "src.stage_plan4_refgen12", "--split", "plan4_timing12", "--limit", "1"])
print("refgen12 first case exit", rc)
path = WORK / "results_stage_plan4_refgen12" / "case_checks.jsonl"
if not path.is_file():
    path = REPO / "results" / "stage_plan4_refgen12" / "run" / "case_checks.jsonl"
rows = [json.loads(l) for l in path.read_text(encoding="utf-8").splitlines() if l.strip()]
for row in rows:
    print(row["example_id"], "ok", row["ok"])
    for k, v in row["checks"].items():
        print("  ", k, {kk: v.get(kk) for kk in ("ok", "mean_kl", "argmax_agree", "labels_equal", "n_steps")} if isinstance(v, dict) else v)
    print("  comparisons", json.dumps(row["comparisons"], indent=1))
if not all(r["ok"] for r in rows):
    raise RuntimeError("Incremental-path check failed. Do not run the remaining cases.")

In [ ]:
rc = subprocess.call([sys.executable, "-m", "src.stage_plan4_refgen12", "--split", "plan4_timing12"])
print("refgen12 exit", rc)

In [ ]:
src = WORK / "results_stage_plan4_refgen12"
if not (src / "summary.json").is_file():
    src = REPO / "results" / "stage_plan4_refgen12" / "run"
data = json.loads((src / "summary.json").read_text(encoding="utf-8"))
print("cases", data["n_cases"], "of", data["n_expected_cases"], "| instrumentation_ok", data["instrumentation_ok"])
keys = ("labels_correct", "n_labels", "cases_all_twelve_correct", "ineligible_marked_eligible",
        "eligible_marked_ineligible", "parse_ok", "plan_truncated", "answer_truncated", "plan_lines_missing")
for b, block in data["labels"].items():
    print(b, {k: block.get(k) for k in keys})
print("\nfinal choice (separate)")
for b, block in data["final_choice"].items():
    print(b, block)
print("\nreproduction vs K/V swap run", data["reproduction_vs_kvmix12"])
print("self-replay free generation", data["self_replay_free_generation"])
print("intervention vs reference text", data["intervention_vs_reference_text"])
for name, block in data["paired_by_case"].items():
    c = block["labels_correct"]
    print(name, {k: c.get(k) for k in ("total_delta", "ci95_per_case", "cases_increased", "cases_decreased")})
div = dict(data["divergence"])
per_case = div.pop("per_case", {})
print("\ndivergence", json.dumps(div, indent=1))
for eid, d in per_case.items():
    print(eid, d)